In [1]:
!pip install -U langgraph langchain langchain-core langchain-community \
               langchain-openai langchain-experimental \
               wikipedia google-search-results \
               pydantic typing-extensions

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11779 sha256=ab66e78654e7d9ced27783e4108620f8f568f1db7b7e5d6cf93e57a64b475994
  Stored in directory: c:\users\mohankumar mc\appdata\local\pip\cache\wheels\30\24\d3\9d46daa49494cef2f32dc9a8f203aac1f10cec980adba03d1f
Successfully built wikipedia


In [ ]:
# Parallel and Looping of Langgraph done below

In [8]:
# -----------------------------------------
# Imports
# -----------------------------------------
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os
import random

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# -----------------------------------------
# Load API key
# -----------------------------------------
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


# LangSmith key
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API Key: ")

# Enable LangSmith tracing
os.environ["LANGSMITH_TRACING"] = "true"

# Project name in LangSmith dashboard
os.environ["LANGSMITH_PROJECT"] = "langgraph-product-agent-colab"



llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.4,
    max_tokens = 300,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)



In [9]:
# -----------------------------------------
# State
# -----------------------------------------
class State(TypedDict):
    product_name: str
    basic_description: str
    features_benefits: str
    target_audience: str
    seo_keywords: str
    marketing_message: str
    final_description: str
    quality_score: int

# -----------------------------------------
# Nodes
# -----------------------------------------

def generate_basic_description(state: State):
    response = llm.invoke([
        SystemMessage(content="You generate short product descriptions."),
        HumanMessage(content=f"Write a brief description of '{state['product_name']}'.")
    ])
    return {"basic_description": response.content}


# ----------- PARALLEL NODES -----------

def add_features(state: State):
    response = llm.invoke(
        f"List key features and benefits:\n{state['basic_description']}"
    )
    return {"features_benefits": response.content}


def identify_audience(state: State):
    response = llm.invoke(
        f"Who is the ideal target audience for this product?\n{state['basic_description']}"
    )
    return {"target_audience": response.content}


def generate_seo_keywords(state: State):
    response = llm.invoke(
        f"Generate SEO keywords for this product:\n{state['basic_description']}"
    )
    return {"seo_keywords": response.content}


# ----------- MERGE NODE -----------

def create_marketing_message(state: State):
    combined = f"""
Features:
{state['features_benefits']}

Audience:
{state['target_audience']}

SEO:
{state['seo_keywords']}
"""
    response = llm.invoke(
        f"Create a compelling marketing message using:\n{combined}"
    )
    return {"marketing_message": response.content}





In [4]:
# ----------- FINAL POLISH -----------

def polish_final_description(state: State):
    response = llm.invoke(
        f"Polish and finalize:\n{state['marketing_message']}"
    )
    return {"final_description": response.content}


# ----------- QUALITY CHECK (LOOP CONTROL) -----------

def evaluate_quality(state: State):
    """
    Simulated quality scoring.
    In real systems you'd use LLM scoring.
    """
    score = random.randint(5, 7)
    print(f"\nQuality Score: {score}")
    return {"quality_score": score}


def improve_description(state: State):
    response = llm.invoke(
        f"Improve this product description to make it more persuasive:\n{state['final_description']}"
    )
    return {"final_description": response.content}




In [5]:

# -----------------------------------------
# Build Graph
# -----------------------------------------
def build_workflow():
    workflow = StateGraph(State)
    # Nodes
    workflow.add_node("basic", generate_basic_description)
    workflow.add_node("features", add_features)
    workflow.add_node("audience", identify_audience)
    workflow.add_node("seo", generate_seo_keywords)
    workflow.add_node("marketing", create_marketing_message)
    workflow.add_node("final", polish_final_description)
    workflow.add_node("evaluate", evaluate_quality)
    workflow.add_node("improve", improve_description)
    # Flow
    workflow.add_edge(START, "basic")
    # ---- PARALLEL FAN OUT ----
    workflow.add_edge("basic", "features")
    workflow.add_edge("basic", "audience")
    workflow.add_edge("basic", "seo")
    # ---- FAN IN (merge) ----
    workflow.add_edge("features", "marketing")
    workflow.add_edge("audience", "marketing")
    workflow.add_edge("seo", "marketing")
    workflow.add_edge("marketing", "final")
    # ---- LOOP SECTION ----
    workflow.add_edge("final", "evaluate")
    workflow.add_conditional_edges(
        "evaluate",
        lambda state: "improve" if state["quality_score"] < 6 else "end",
        {
            "improve": "improve",
            "end": END
        }
    )
    workflow.add_edge("improve", "evaluate")
    return workflow.compile()


In [10]:
# -----------------------------------------
# Run
# -----------------------------------------
if __name__ == "__main__":
    app = build_workflow()

    initial_state: State = {
        "product_name": "Smart Water Bottle",
        "basic_description": "",
        "features_benefits": "",
        "target_audience": "",
        "seo_keywords": "",
        "marketing_message": "",
        "final_description": "",
        "quality_score": 0
    }

    result = app.invoke(initial_state)

    print("\nFINAL OUTPUT:\n")
    print(result["final_description"])


Quality Score: 7

FINAL OUTPUT:

and smartphone alerts, you'll stay motivated to meet your hydration goals every day.

#### **Join the Hydration Revolution!**
Don’t let dehydration hold you back. Embrace the future of hydration with the **Smart Water Bottle** and transform the way you drink water. Whether you’re hitting the gym, working at your desk, or exploring the great outdoors, this innovative bottle is your perfect companion.

### **Get Yours Today!**
Stay hydrated, stay healthy, and take the first step toward better wellness with the **Smart Water Bottle**. Order now and experience the difference!


In [7]:
# =========================================================
# GRADIO UI FOR SECOND LANGGRAPH AGENTIC WORKFLOW
# Shows output of each agent/node step
# =========================================================

!pip install -q gradio

import gradio as gr

# Build the already-defined workflow
app = build_workflow()


def run_product_agent(product_name):
    initial_state: State = {
        "product_name": product_name,
        "basic_description": "",
        "features_benefits": "",
        "target_audience": "",
        "seo_keywords": "",
        "marketing_message": "",
        "final_description": "",
        "quality_score": 0
    }

    result = app.invoke(initial_state)

    basic = result.get("basic_description", "")
    features = result.get("features_benefits", "")
    audience = result.get("target_audience", "")
    seo = result.get("seo_keywords", "")
    marketing = result.get("marketing_message", "")
    final_description = result.get("final_description", "")
    quality_score = result.get("quality_score", "")

    process_view = f"""
# LangGraph Agentic Product Description Process

## 1. Basic Description Agent
{basic}

---

## 2. Features & Benefits Agent
{features}

---

## 3. Target Audience Agent
{audience}

---

## 4. SEO Keywords Agent
{seo}

---

## 5. Marketing Message Agent
{marketing}

---

## 6. Final Polish Agent
{final_description}

---

## 7. Quality Evaluation Agent
**Quality Score:** {quality_score}

---

## Final Output
{final_description}
"""

    return (
        basic,
        features,
        audience,
        seo,
        marketing,
        str(quality_score),
        final_description,
        process_view
    )


with gr.Blocks(title="LangGraph Product Description Agent") as demo:
    gr.Markdown("# LangGraph Product Description Agent")
    gr.Markdown(
        "Enter a product name and see how each LangGraph agent transforms it step by step."
    )

    with gr.Row():
        product_input = gr.Textbox(
            label="Product Name",
            value="Smart Water Bottle",
            placeholder="Enter product name..."
        )

    run_button = gr.Button("Generate Product Description")

    with gr.Tab("Step-by-Step Agent Outputs"):
        basic_output = gr.Textbox(
            label="1. Basic Description Agent",
            lines=6
        )

        features_output = gr.Textbox(
            label="2. Features & Benefits Agent",
            lines=8
        )

        audience_output = gr.Textbox(
            label="3. Target Audience Agent",
            lines=8
        )

        seo_output = gr.Textbox(
            label="4. SEO Keywords Agent",
            lines=6
        )

        marketing_output = gr.Textbox(
            label="5. Marketing Message Agent",
            lines=8
        )

        quality_output = gr.Textbox(
            label="6. Quality Score"
        )

        final_output = gr.Textbox(
            label="7. Final Product Description",
            lines=10
        )

    with gr.Tab("Full Process View"):
        full_process_output = gr.Markdown()

    run_button.click(
        fn=run_product_agent,
        inputs=product_input,
        outputs=[
            basic_output,
            features_output,
            audience_output,
            seo_output,
            marketing_output,
            quality_output,
            final_output,
            full_process_output
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



Quality Score: 7
